## 0.0 Import necessary libraries
i'll be using PyTorch for this assignment so we'll be using `torch`
also i changed the folder names of `Positive -> 1` and `Negative -> 0` for easier access
i'll also add them into a parent folder called `Data`

i will train my model only on about a 1000 of each class, because no matter what i did, i couldn't add the whole dataset into the notebook

In [19]:
# Settng up necessary libraries
import numpy as np
import matplotlib.pyplot
from torch.utils.data import DataLoader
from pathlib import Path
import torch
from torch import nn
import torchvision
from torchvision import datasets
from torchvision.transforms import Compose
from torchvision.transforms import ToTensor
from torchvision import models

# Setting up device-agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
import os
from torch.utils.data import random_split, DataLoader
from torchvision import datasets, transforms
import shutil

dataset_dir = '/content/Data'

# Data Augmentation for the images
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])
# Whatever this is because without this my code will crash LOL
shutil.rmtree(os.path.join(dataset_dir, '.ipynb_checkpoints'), ignore_errors=True)

# Load the dataset using ImageFolder
full_dataset = datasets.ImageFolder(root=dataset_dir, transform=data_transform)

# split ratio, 80% train, 20% test
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

# Perform the split
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# Default Batch_size
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# getting to know our data better
print(f"Training dataset size: {len(train_dataset)}")
print(f"Testing dataset size: {len(test_dataset)}")


Training dataset size: 1600
Testing dataset size: 400


In [8]:
# Checking out to see if anything about the shape is wrong
img, label = next(iter(train_loader))
img.shape, label.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32]))

Everything looks fine

## 1.0 Creating a Custom CNN model

In [9]:
class CustomModel(nn.Module):
    def __init__(self, input_shape, hidden_units, output_shape):
        super().__init__()
        self.conv_layer_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size = 2, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size = 2, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2))
        self.conv_layer_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size = 2, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size = 2, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2))
        self.conv_layer_3 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size = 2, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size = 2, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2))
        self.conv_layer_4 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size = 2, stride=1, padding=0),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size = 2, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=int(hidden_units*54*54), out_features=output_shape)
        )
    def forward(self, x:torch.Tensor):
        x = self.conv_layer_1(x)
        x = self.conv_layer_2(x)
        x = self.classifier(x)
        return x

In [11]:
model_0 = CustomModel(input_shape=3, hidden_units=10, output_shape=2).to(device)
model_0

CustomModel(
  (conv_layer_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(2, 2), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(2, 2), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_layer_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(2, 2), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(2, 2), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_layer_3): Sequential(
    (0): Conv2d(10, 10, kernel_size=(2, 2), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(2, 2), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_layer_4): Sequential(
    (0): Conv2d(10, 10, kernel_size=(2, 2), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(2, 2), stride=(1, 1))
    (3): ReLU()
    

## 1.1 Getting a summary of our model

In [12]:
# install torchinfo
!pip install torchinfo

In [13]:
from torchinfo import summary
summary(model_0)

Layer (type:depth-idx)                   Param #
CustomModel                              --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       130
│    └─ReLU: 2-2                         --
│    └─Conv2d: 2-3                       410
│    └─ReLU: 2-4                         --
│    └─MaxPool2d: 2-5                    --
├─Sequential: 1-2                        --
│    └─Conv2d: 2-6                       410
│    └─ReLU: 2-7                         --
│    └─Conv2d: 2-8                       410
│    └─ReLU: 2-9                         --
│    └─MaxPool2d: 2-10                   --
├─Sequential: 1-3                        --
│    └─Conv2d: 2-11                      410
│    └─ReLU: 2-12                        --
│    └─Conv2d: 2-13                      410
│    └─ReLU: 2-14                        --
│    └─MaxPool2d: 2-15                   --
├─Sequential: 1-4                        --
│    └─Conv2d: 2-16                      410
│    └─ReLU: 2-17   

## 2.0 Creating training and testing functions to make our work easier

In [14]:
def train_step(model:torch.nn.Module,
               dataloader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device=device):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss +=loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(y_pred, dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)
    train_loss = train_loss/len(dataloader)
    train_acc = train_acc/len(dataloader)
    return train_loss, train_acc

In [15]:
def test_step(model:torch.nn.Module,
              dataloader:torch.utils.data.DataLoader,
              loss_fn:torch.nn.Module,
              device=device):
    model.eval()
    test_loss, test_acc = 0, 0
    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            test_pred = model(X)
            test_loss += loss_fn(test_pred, y).item()
            test_pred_class = torch.argmax(test_pred, dim=1)
            test_acc += (test_pred_class == y).sum().item()/len(test_pred)
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [16]:
from tqdm.auto import tqdm
def train(model:torch.nn.Module,
          train_dataloader:torch.utils.data.DataLoader,
          test_dataloader:torch.utils.data.DataLoader,
          optimizer:torch.optim.Optimizer,
          loss_fn:torch.nn.Module = nn.CrossEntropyLoss(),
          epochs:int = 10,
          device=device):
    # 1. Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []}
    # 3. Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)
        test_loss, test_acc = test_step(model=model,
                                         dataloader=test_dataloader,
                                         loss_fn=loss_fn,
                                         device=device)
        # Print out whats happening

        print(f"Epoch: {epoch} | Train loss : {train_loss:.4f} | Train acc: {train_acc:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")
        # Update our results
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
    # Return the filled results at then of the epoch
    return results

## 3.0 Training our Data on our custom model

In [17]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
NUM_EPOCHS = 5

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.001)

model_0_results = train(model=model_0,
                       train_dataloader=train_loader,
                       test_dataloader=test_loader,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       epochs=NUM_EPOCHS)

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 0 | Train loss : 0.6306 | Train acc: 0.6531 | Test loss: 0.4632 | Test acc: 0.7933
Epoch: 1 | Train loss : 0.3412 | Train acc: 0.8762 | Test loss: 0.2051 | Test acc: 0.9495
Epoch: 2 | Train loss : 0.1828 | Train acc: 0.9413 | Test loss: 0.1387 | Test acc: 0.9519
Epoch: 3 | Train loss : 0.1592 | Train acc: 0.9531 | Test loss: 0.1045 | Test acc: 0.9712
Epoch: 4 | Train loss : 0.1391 | Train acc: 0.9644 | Test loss: 0.0704 | Test acc: 0.9760


around 97% test accuracy !! let's see if we can get that closer to a 100%!!

## 3.1 Using a pretrained model to see if we get better results

In [20]:
# I want to use the model that helped me create my foodvisionmini app on huggingface :)

# Setting the weights and the model
effnet_b2_weights = models.EfficientNet_B0_Weights.DEFAULT
effnet_b2_model = models.efficientnet_b0(weights=effnet_b2_weights).to(device)

# Changing our output value : 1000 -> 2
effnet_b2_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=2, bias=True)
).to(device) # Sending everything to cuda
optimizer = torch.optim.SGD(params=effnet_b2_model.parameters(), lr=0.001) # Setting up optimizer
effnet_b2_model

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 68.6MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

## 3.2 Getting a summary of our model

In [21]:
summary(effnet_b2_model)

Layer (type:depth-idx)                                  Param #
EfficientNet                                            --
├─Sequential: 1-1                                       --
│    └─Conv2dNormActivation: 2-1                        --
│    │    └─Conv2d: 3-1                                 864
│    │    └─BatchNorm2d: 3-2                            64
│    │    └─SiLU: 3-3                                   --
│    └─Sequential: 2-2                                  --
│    │    └─MBConv: 3-4                                 1,448
│    └─Sequential: 2-3                                  --
│    │    └─MBConv: 3-5                                 6,004
│    │    └─MBConv: 3-6                                 10,710
│    └─Sequential: 2-4                                  --
│    │    └─MBConv: 3-7                                 15,350
│    │    └─MBConv: 3-8                                 31,290
│    └─Sequential: 2-5                                  --
│    │    └─MBConv: 3-9         

## 3.3 Training our model
this time we're going for higher epochs also !

In [22]:
effnetb2_results = train(model=effnet_b2_model,
                         train_dataloader = train_loader,
                         test_dataloader = test_loader,
                         optimizer=optimizer,
                         loss_fn=loss_fn,
                         epochs = 10,
                         device=device)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 0 | Train loss : 0.6278 | Train acc: 0.6744 | Test loss: 0.5962 | Test acc: 0.7692
Epoch: 1 | Train loss : 0.5292 | Train acc: 0.8237 | Test loss: 0.4665 | Test acc: 0.9014
Epoch: 2 | Train loss : 0.4561 | Train acc: 0.9056 | Test loss: 0.3892 | Test acc: 0.9567
Epoch: 3 | Train loss : 0.3925 | Train acc: 0.9319 | Test loss: 0.3366 | Test acc: 0.9519
Epoch: 4 | Train loss : 0.3440 | Train acc: 0.9506 | Test loss: 0.2916 | Test acc: 0.9543
Epoch: 5 | Train loss : 0.2985 | Train acc: 0.9688 | Test loss: 0.2558 | Test acc: 0.9760
Epoch: 6 | Train loss : 0.2657 | Train acc: 0.9681 | Test loss: 0.2201 | Test acc: 0.9784
Epoch: 7 | Train loss : 0.2413 | Train acc: 0.9712 | Test loss: 0.1994 | Test acc: 0.9880
Epoch: 8 | Train loss : 0.2180 | Train acc: 0.9769 | Test loss: 0.1720 | Test acc: 0.9856
Epoch: 9 | Train loss : 0.1960 | Train acc: 0.9738 | Test loss: 0.1573 | Test acc: 0.9904


99% !!! i'd say that's close to perfection.